In [1]:
import os
from dotenv import load_dotenv

API Keys

In [2]:
load_dotenv()

True

Embedding model from hugging face

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\saza\.conda\envs\langchain\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM from GROQ

In [4]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama3-70b-8192")

VectorStore

In [5]:
from langchain_chroma import Chroma
vectorstore = Chroma(persist_directory="chroma_legal_db", embedding_function=embedding_model)

Retriver

In [6]:
retriever = vectorstore.as_retriever()

Prompt

In [7]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an Intelligent chatbot when a user inputs the document you can identify the missing terms or non complient clauses based on the given context and help the user to fix those.If you dont have the knowledge in the paticular area to check the input say that you dont know."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}"),
    ]
)

Rag chain

In [8]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# question answer chain
qa_chain = create_stuff_documents_chain(llm,prompt)

# rag chain
rag_chain = create_retrieval_chain(retriever,qa_chain)

In [11]:
import os
from langchain.document_loaders import PyPDFLoader

pdf_dir = "./legal_docs"


loader = PyPDFLoader(os.path.join(pdf_dir, 'agreement.pdf'))
documents = loader.load()

In [14]:
user_input = ""

for i in documents:
    user_input = user_input + i.page_content

print(user_input)

SHAREHOLDERS’ AGREEMENT 
THIS AGREEMENT is made and entered into on this ___ day of __________ 20__. 
BY AND BETWEEN 
1. [Company Name (Pvt) Ltd], a company duly incorporated under the Companies Act No. 7 of 
2007 of Sri Lanka and having its registered office at [address] (hereinafter referred to as the 
“Company”); 
AND 
2. [Shareholder 1’s Full Name], of [address], holding [__]% of the issued shares of the 
Company (hereinafter referred to as “Shareholder 1”); 
3. [Shareholder 2’s Full Name], of [address], holding [__]% of the issued shares of the 
Company (hereinafter referred to as “Shareholder 2”); 
(Each a “Party” and collectively referred to as the “Parties”). 
 
WHEREAS: 
• The Company is a duly incorporated private limited liability company under the laws of Sri 
Lanka; 
• The Shareholders desire to set forth their respective rights, obligations, and liabilities in 
respect of their shareholding and management of the Company. 
 
1. INTERPRETATION 
1.1 Definitions 
In this Agre

In [15]:
response = rag_chain.invoke({"input":user_input})
print(response["answer"])

I've reviewed the provided shareholders' agreement, and I'll highlight some potential issues and suggestions for improvement:

1. **Missing essential clauses**: The agreement lacks provisions related to:
	* Share valuation and dispute resolution mechanisms.
	* Drag-along and tag-along rights.
	* Voting rights and quorum requirements.
	* Confidentiality and non-disclosure obligations.
	* Dispute resolution mechanisms for deadlocks and disputes beyond mediation and arbitration.
2. **Unclear definitions**: Some definitions, such as "Act" and "Board," are not explicitly stated as referring to the Companies Act No. 7 of 2007 and the Board of Directors of the Company, respectively.
3. **Inconsistent formatting and numbering**: The agreement's formatting and numbering are inconsistent, making it difficult to follow.
4. **Lack of clarity on governance and board composition**: The agreement does not specify the process for appointing or removing directors, or the role of the Board in managing t